# EDA Bronze — exploración de datos crudos

Análisis exploratorio de los **2 datasets que ingesta la capa bronze** (`pipeline/assets/bronze.py`)
para diseñar las transformaciones de **silver/gold** y los **chequeos de calidad** (ADRs 0013-0015).

- **Motor:** DuckDB (SQL-first, alineado con dbt) + pandas para resúmenes.
- **Fuente:** CSV locales en `../data/` (mismos bytes que el bronze; sin credenciales).
- **Entorno:** conda `eda-bronze` (`environment.yml`). NO usa el poetry de producción.
- **Out of scope:** `produccin-...-2026.csv` (convencional) NO es parte del bronze.

Los hallazgos destilados viven en `../docs/data/eda-bronze.md`.

In [ ]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 200)

DATA = '../data'
LISTADO = f'{DATA}/listado-de-pozos-cargados-por-empresas-operadoras.csv'
PROD = f'{DATA}/produccin-de-pozos-de-gas-y-petrleo-no-convencional.csv'

con = duckdb.connect()
# normalize_names + el lector de DuckDB limpian el BOM del header (utf-8-sig).
con.execute(f"CREATE VIEW listado AS SELECT * FROM read_csv_auto('{LISTADO}', normalize_names=true)")
con.execute(f"CREATE VIEW prod AS SELECT * FROM read_csv_auto('{PROD}', normalize_names=true)")
print('Vistas creadas: listado, prod')

## 1. listado_pozos — schema, grano y nulos

In [ ]:
print('shape:', con.sql('SELECT count(*) FROM listado').fetchone()[0], 'filas')
con.sql('DESCRIBE listado').df()

In [ ]:
# Grano: idpozo unico, sin nulos, sin duplicados?
con.sql('''
    SELECT count(*) AS filas,
           count(DISTINCT idpozo) AS idpozo_distintos,
           count(*) FILTER (WHERE idpozo IS NULL) AS idpozo_nulos
    FROM listado
''').df()

In [ ]:
# % de nulos por columna (top 20)
lp = con.sql('SELECT * FROM listado').df()
(lp.isna().mean().mul(100).round(1).sort_values(ascending=False).head(20)
   .rename('pct_nulos').to_frame())

In [ ]:
# Categoricas candidatas a dimension
for col in ['cuenca', 'provincia', 'tipo_reservorio', 'clasificacion', 'gasplus']:
    print(f'\n== {col} ==')
    print(con.sql(f'SELECT {col}, count(*) AS n FROM listado GROUP BY 1 ORDER BY n DESC LIMIT 8').df().to_string(index=False))

In [ ]:
# Outliers numericos (profundidad, coordenadas)
con.sql('''
    SELECT
        min(profundidad) AS prof_min, max(profundidad) AS prof_max, avg(profundidad) AS prof_avg,
        min(coordenadax) AS x_min, max(coordenadax) AS x_max,
        min(coordenaday) AS y_min, max(coordenaday) AS y_max
    FROM listado
''').df()

## 2. produccion_no_convencional — grano y duplicados

In [ ]:
print('shape:', con.sql('SELECT count(*) FROM prod').fetchone()[0], 'filas')
# Grano (idpozo, anio, mes): duplicados?
con.sql('''
    SELECT count(*) AS filas,
           count(DISTINCT (idpozo, anio, mes)) AS grano_distinto,
           count(*) - count(DISTINCT (idpozo, anio, mes)) AS duplicados_grano,
           min(anio) AS anio_min, max(anio) AS anio_max
    FROM prod
''').df()

In [ ]:
# rectificado: flag de correccion (clave para el tipo de carga, ADR-0013)
con.sql('SELECT rectificado, count(*) AS n FROM prod GROUP BY 1 ORDER BY n DESC').df()

In [ ]:
# Columnas constantes / casi vacias -> no promover a gold
pr = con.sql('SELECT * FROM prod').df()
print('habilitado:'); print(pr['habilitado'].value_counts(dropna=False))
print('\ntipo_de_recurso:'); print(pr['tipo_de_recurso'].value_counts(dropna=False))
print('\n% nulos (top 10):')
print(pr.isna().mean().mul(100).round(1).sort_values(ascending=False).head(10))

In [ ]:
# Categoricas: estado, tipo de pozo, extraccion, recurso
for col in ['tipoestado', 'tipopozo', 'tipoextraccion', 'sub_tipo_recurso']:
    print(f'\n== {col} ==')
    print(con.sql(f'SELECT {col}, count(*) AS n FROM prod GROUP BY 1 ORDER BY n DESC LIMIT 8').df().to_string(index=False))

In [ ]:
# Medidas de la fact: rangos, ceros y negativos (errores de fuente)
con.sql('''
    SELECT 'prod_pet' AS medida, min(prod_pet) AS mn, max(prod_pet) AS mx, avg(prod_pet) AS avg,
           count(*) FILTER (WHERE prod_pet < 0) AS negativos, count(*) FILTER (WHERE prod_pet = 0) AS ceros FROM prod
    UNION ALL SELECT 'prod_gas', min(prod_gas), max(prod_gas), avg(prod_gas),
           count(*) FILTER (WHERE prod_gas < 0), count(*) FILTER (WHERE prod_gas = 0) FROM prod
    UNION ALL SELECT 'prod_agua', min(prod_agua), max(prod_agua), avg(prod_agua),
           count(*) FILTER (WHERE prod_agua < 0), count(*) FILTER (WHERE prod_agua = 0) FROM prod
    UNION ALL SELECT 'profundidad', min(profundidad), max(profundidad), avg(profundidad),
           count(*) FILTER (WHERE profundidad < 0), count(*) FILTER (WHERE profundidad = 0) FROM prod
''').df()

## 3. Integridad referencial produccion.idpozo → listado.idpozo

In [ ]:
# Pozos en produccion sin match en listado (huerfanos) -> test relationships en warn
con.sql('''
    WITH huerfanos AS (
        SELECT DISTINCT p.idpozo
        FROM prod p LEFT JOIN listado l ON p.idpozo = l.idpozo
        WHERE l.idpozo IS NULL
    )
    SELECT (SELECT count(DISTINCT idpozo) FROM prod) AS pozos_en_prod,
           (SELECT count(DISTINCT idpozo) FROM listado) AS pozos_en_listado,
           (SELECT count(*) FROM huerfanos) AS pozos_huerfanos,
           (SELECT count(*) FROM prod WHERE idpozo IN (SELECT idpozo FROM huerfanos)) AS filas_huerfanas
''').df()

## Conclusiones

Ver el detalle y el mapeo a transformaciones silver, modelo estrella y tests de calidad en
[`../docs/data/eda-bronze.md`](../docs/data/eda-bronze.md).